# 🦅 PREDATOR Analytics v56.1.4-ELITE — Full Stack Deploy on Colab (ZROK Edition)

**Стек**: PostgreSQL 16 + Redis 7 + FastAPI + Nginx + **ZROK Tunnel**

**Порядок запуску**: Виконайте клітинки **зверху вниз**.


In [ ]:
%%bash
# 1. Системні залежності
echo '📦 Встановлення системних пакетів...'
apt-get update -qq
apt-get install -y -qq postgresql-16 postgresql-client-16 redis-server nginx curl wget 2>&1 | tail -5

echo '📦 Zrok тунель...'
wget -q https://github.com/openziti/zrok/releases/download/v0.4.45/zrok_0.4.45_linux_amd64.tar.gz -O /tmp/zrok.tar.gz
tar -xzf /tmp/zrok.tar.gz --wildcards '*/zrok' -O > /usr/local/bin/zrok
chmod +x /usr/local/bin/zrok
zrok disable 2>/dev/null || true  # Disable active if any
zrok enable 1eeje4um7yvA
echo '✅ Система готова!'

In [ ]:
%%bash
# 2. Бази даних (PostgreSQL та Redis)
service postgresql start
service redis-server start
sleep 3
su -c "psql -c \"CREATE USER predator WITH PASSWORD 'predator_secret_2026';\"" postgres 2>/dev/null || true
su -c "psql -c \"CREATE DATABASE predator_db OWNER predator;\"" postgres 2>/dev/null || true
su -c "psql -d predator_db" postgres << 'SQL'
CREATE TABLE IF NOT EXISTS companies (id BIGSERIAL PRIMARY KEY, edrpou VARCHAR(10) UNIQUE, name TEXT, risk_score DECIMAL(5,2) DEFAULT 0, risk_level VARCHAR(20) DEFAULT 'stable', status VARCHAR(20) DEFAULT 'active', sector VARCHAR(100), created_at TIMESTAMPTZ DEFAULT NOW());
CREATE TABLE IF NOT EXISTS declarations (id BIGSERIAL PRIMARY KEY, declaration_number VARCHAR(50) UNIQUE, declaration_date DATE, company_name TEXT, company_edrpou VARCHAR(10), product_code VARCHAR(20), product_name TEXT, value_usd DECIMAL(15,2), weight_kg DECIMAL(15,3), created_at TIMESTAMPTZ DEFAULT NOW());
INSERT INTO companies (edrpou, name, risk_score, risk_level, sector) VALUES ('38210342', 'ТОВ ЕНЕРДЖИ-ГРУП', 92, 'critical', 'Енергетика'), ('41092384', 'ПРАТ ТЕХНО-ВЕСТ', 75, 'high', 'Технології') ON CONFLICT DO NOTHING;
SQL
echo '✅ Бази даних готові!'

In [ ]:
%%bash
# 3. Запуск FastAPI
pip install -q fastapi uvicorn[standard] sqlalchemy asyncpg psycopg2-binary redis httpx
mkdir -p /opt/predator-api
cat > /opt/predator-api/main.py << 'PYEOF'
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import psycopg2
from datetime import datetime

app = FastAPI(title="PREDATOR Analytics API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def root(): return {"status": "PREDATOR Analytics v56.1.4-COLAB", "timestamp": datetime.now().isoformat()}
@app.get("/api/v1/azr/status")
def status(): return {"status": "operational", "mode": "colab", "timestamp": datetime.now().isoformat()}
@app.get("/health")
def health(): return {"status": "ok", "version": "56.1.4-COLAB"}
@app.post("/api/v1/auth/login")
def login(credentials: dict):
    if credentials.get("username") == "admin":
        return {"access_token": "colab-token-admin", "token_type": "bearer", "user": {"username": "admin", "role": "admin"}}
    raise HTTPException(status_code=401, detail="Невірні дані")
PYEOF
cd /opt/predator-api
pkill -f uvicorn || true
nohup uvicorn main:app --host 0.0.0.0 --port 8000 --workers 2 > /var/log/predator-api.log 2>&1 &
echo '✅ FastAPI запущено!'

In [ ]:
%%bash
# 4. Налаштування Nginx як проксі (API + заглушка UI)
mkdir -p /opt/predator-ui
cat > /opt/predator-ui/index.html << 'HTML'
<!DOCTYPE html><html lang="uk"><head><meta charset="UTF-8"><title>PREDATOR Analytics</title>
<style>body{background:#020408;color:#fff;font-family:system-ui;text-align:center;padding:10vh;} a{color:#dc2626;}</style></head>
<body><h1>🦅 PREDATOR Analytics API (Colab)</h1>
<p>Бекенд успішно запущено. Будь ласка, використовуйте <b>UI на Mac</b> для роботи.</p>
<p>Перевірка з'єднання: <a href="/api/v1/azr/status">/api/v1/azr/status</a></p></body></html>
HTML

cat > /etc/nginx/sites-available/predator << 'NGINX'
server {
    listen 3030;
    root /opt/predator-ui;
    index index.html;
    location /api/ {
        proxy_pass http://localhost:8000;
        proxy_set_header Host $host;
        add_header 'Access-Control-Allow-Origin' '*';
    }
    location /health { proxy_pass http://localhost:8000/health; }
    location / { try_files $uri $uri/ /index.html; }
}
NGINX
ln -sf /etc/nginx/sites-available/predator /etc/nginx/sites-enabled/predator
rm -f /etc/nginx/sites-enabled/default
nginx -t && service nginx restart
echo '✅ Nginx запущено!'

In [ ]:
# 5. Запуск тунелю ZROK
import subprocess, time, re

print('🌐 Запуск ZROK тунелю...')
subprocess.run('pkill -f zrok', shell=True)
subprocess.run('rm -f /tmp/zrok.log', shell=True)

zrok_proc = subprocess.Popen(
    ['nohup', 'zrok', 'share', 'public', 'http://localhost:3030', '--headless'],
    stdout=open('/tmp/zrok.log', 'w'),
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    try:
        with open('/tmp/zrok.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-z0-9\-]*\.share\.zrok\.io', content)
            if match:
                public_url = match.group(0)
                break
    except:
        pass
    time.sleep(1)

if public_url:
    print('\n' + '='*65)
    print('🦅 PREDATOR Analytics — COLAB DEPLOYED (ZROK)!')
    print('='*65)
    print(f'\n🌐 Публічний URL: {public_url}')
    print(f'⚙️ API URL:       {public_url}/api/v1')
    print('\n' + '─'*65)
    print('👇 ВСТАВТЕ ЦЕ В .env.local НА MAC:')
    print(f'VITE_API_URL={public_url}/api/v1')
    print(f'VITE_BACKEND_PROXY_TARGET={public_url}')
    print('VITE_ENABLE_MOCK_API=false')
    print('VITE_MODE=remote')
    print('='*65)
else:
    print('❌ Не вдалося отримати zrok URL. Логи:')
    with open('/tmp/zrok.log', 'r') as f:
        print(f.read())

In [ ]:
# 6. Кеп-елайв сесії
import time, datetime, httpx
print(f'🦅 Gateway активний на {public_url}')
while True:
    try: httpx.get('http://localhost:8000/health', timeout=3); status = '✅'
    except: status = '⚠️'
    print(f'\r⏱️ {datetime.datetime.now().strftime("%H:%M:%S")} | API {status}', end='', flush=True)
    time.sleep(60)